# Project 30 — BioAge AI: Biological Age & Longevity Risk Intelligence

## Biomarker-derived age proxy for longevity analytics

This project uses **real NHANES biomarker data** to learn an age-related physiological signal from routine blood and cardiovascular measurements. It is deliberately framed as a **research/portfolio longevity model**, not a medical diagnostic tool and not evidence that any treatment slows human aging.

**Core question:** Can routinely measured biomarkers recover a meaningful age-related signal, and which measurements carry the most predictive information?


### Data provenance and scientific positioning

- Source family: **NHANES**, the U.S. National Health and Nutrition Examination Survey.
- Working table: public NHANES-derived biomarker table assembled for an aging-biomarker analysis.
- Independent reference: the UCI NHANES age-prediction subset documents the same use case of physiological, lifestyle and biochemical measures for age prediction.
- Contemporary research context: large-scale biological-age research continues to use NHANES alongside other population cohorts.

**Important:** predicted age here is a **biomarker-derived age proxy**. It is not equivalent to validated PhenoAge/KDM/epigenetic clocks and should not be interpreted as a diagnosis of accelerated aging.


## 1. Load, clean and define the adult cohort


In [1]:
# Deterministic compact gradient-boosted regression tree implementation
import math, numpy as np, pandas as pd

DATA_URL = "https://raw.githubusercontent.com/epenning/aging-biomarkers/main/nhanes_data.csv"
FEATURES = ["Albumin","Blood_Urea_Nitrogen","Creatinine","C_Reactive_Protein","Glucose","Glycohemoglobin","Total_Cholesterol","Uric_Acid","White_Blood_Cell_Count","Lymphocyte_Percent","Mean_Cell_Volume","Red_Cell_Distribution_Width","Systolic_Blood_Pressure"]
MODEL_FEATURES = FEATURES + ["Male"]

df = pd.read_csv(DATA_URL)
df = df[df["Age"] >= 18].copy()
for c in FEATURES + ["Age"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["Male"] = (df["Gender"] == "Male").astype(float)
df = df.dropna(subset=FEATURES + ["Age", "Male"]).reset_index(drop=True)

# Reproduce the same deterministic split used for the verified outputs.
def lcg_keys(n, seed=30):
    x = seed & 0xffffffff
    out = []
    for _ in range(n):
        x = (1664525*x + 1013904223) & 0xffffffff
        out.append(x / 4294967296.0)
    return np.array(out)

df = df.assign(_k=lcg_keys(len(df))).sort_values("_k").drop(columns="_k").reset_index(drop=True)
n = len(df); ntr = int(.70*n); nv = int(.15*n)
train_df, val_df, test_df = df.iloc[:ntr], df.iloc[ntr:ntr+nv], df.iloc[ntr+nv:]
print(f"Rows in source table: {len(pd.read_csv(DATA_URL))}")
print(f"Adult complete-case rows used: {len(df)}")
print(f"Features: {len(MODEL_FEATURES)}")
print(f"Train / validation / test: {len(train_df)} / {len(val_df)} / {len(test_df)}")


Rows in source table: 5251
Adult complete-case rows used: 4552
Features: 14
Train / validation / test: 3186 / 682 / 684


## 2. Deterministic gradient-boosted regression trees from first principles

The model is implemented directly rather than hidden behind a library call. This makes the portfolio evidence interview-defensible: candidate thresholds, squared-error splits, shrinkage, early model selection and held-out evaluation are explicit.


In [2]:
class Node:
    def __init__(self, leaf=False, value=None, feature=None, threshold=None, left=None, right=None):
        self.leaf=leaf; self.value=value; self.feature=feature; self.threshold=threshold; self.left=left; self.right=right

def _leaf(indices, resid):
    return Node(leaf=True, value=float(np.mean(resid[indices])))

def _build_tree(X, resid, indices, depth=0, max_depth=3, min_leaf=35):
    if depth >= max_depth or len(indices) < 2*min_leaf:
        return _leaf(indices, resid)
    best = None
    for j in range(X.shape[1]):
        order = indices[np.argsort(X[indices, j], kind="mergesort")]
        vals = X[order, j]; rr = resid[order]
        ps = np.r_[0.0, np.cumsum(rr)]; ps2 = np.r_[0.0, np.cumsum(rr*rr)]
        m = len(order)
        for c in range(1,24):
            k = int(math.floor(c*m/24))
            if k < min_leaf or m-k < min_leaf or vals[k-1] == vals[k]:
                continue
            ln, rn = k, m-k; ls, rs = ps[k], ps[m]-ps[k]; ls2, rs2 = ps2[k], ps2[m]-ps2[k]
            sse = (ls2-ls*ls/ln)+(rs2-rs*rs/rn)
            if best is None or sse < best[0]:
                best = (sse, j, (vals[k-1]+vals[k])/2)
    if best is None: return _leaf(indices, resid)
    _, j, t = best
    left_idx = indices[X[indices,j] <= t]; right_idx = indices[X[indices,j] > t]
    if len(left_idx) < min_leaf or len(right_idx) < min_leaf: return _leaf(indices, resid)
    return Node(False, feature=j, threshold=float(t),
                left=_build_tree(X,resid,left_idx,depth+1,max_depth,min_leaf),
                right=_build_tree(X,resid,right_idx,depth+1,max_depth,min_leaf))

def _tree_predict_one(node, x):
    while not node.leaf:
        node = node.left if x[node.feature] <= node.threshold else node.right
    return node.value

def _tree_predict(node, X):
    return np.array([_tree_predict_one(node,row) for row in X])

def regression_metrics(y,p):
    mae=float(np.mean(np.abs(y-p))); rmse=float(np.sqrt(np.mean((y-p)**2)))
    r2=float(1-np.sum((y-p)**2)/np.sum((y-y.mean())**2))
    return mae,rmse,r2

Xtr=train_df[MODEL_FEATURES].to_numpy(float); ytr=train_df["Age"].to_numpy(float)
Xv=val_df[MODEL_FEATURES].to_numpy(float); yv=val_df["Age"].to_numpy(float)
Xt=test_df[MODEL_FEATURES].to_numpy(float); yt=test_df["Age"].to_numpy(float)
base=float(ytr.mean()); lr=0.06; trees=[]
ptr=np.full(len(ytr),base); pv=np.full(len(yv),base)
best_iter=0; best_mae=float("inf"); best_val=None
for it in range(120):
    resid=ytr-ptr
    tree=_build_tree(Xtr,resid,np.arange(len(ytr)))
    trees.append(tree)
    ptr += lr*_tree_predict(tree,Xtr); pv += lr*_tree_predict(tree,Xv)
    mm=regression_metrics(yv,pv)
    if mm[0] < best_mae:
        best_mae=mm[0]; best_iter=it+1; best_val=mm

pt=np.full(len(yt),base)
for tree in trees[:best_iter]: pt += lr*_tree_predict(tree,Xt)
test_mae,test_rmse,test_r2=regression_metrics(yt,pt)
print(f"Best trees: {best_iter}")
print(f"Learning rate: {lr}")
print(f"Validation MAE: {best_val[0]:.3f} years")
print(f"Test MAE: {test_mae:.3f} years")
print(f"Test RMSE: {test_rmse:.3f} years")
print(f"Test R²: {test_r2:.3f}")


Best trees: 120
Learning rate: 0.06
Validation MAE: 9.453 years
Test MAE: 9.736 years
Test RMSE: 11.993 years
Test R²: 0.496


## 3. Held-out performance

The test set is never used to choose the number of boosting rounds. The final model explains roughly half of held-out chronological-age variance from the available biomarker panel, with an error of about ten years. That is useful as a **signal-discovery benchmark**, but not accurate enough to claim an individual clinical biological age.

![Chronological vs predicted age](docs/images/project30_bioage/age_vs_predicted.svg)


## 4. Which biomarkers drive the predictive signal?


In [3]:
split_counts = np.zeros(len(MODEL_FEATURES), dtype=int)
def count_splits(node):
    if node.leaf: return
    split_counts[node.feature] += 1
    count_splits(node.left); count_splits(node.right)
for tree in trees[:best_iter]: count_splits(tree)
importance = pd.DataFrame({"feature": MODEL_FEATURES, "split_count": split_counts}).sort_values("split_count", ascending=False)
print("\n".join(f"{i+1}. {row.feature}: {int(row.split_count)} splits" for i,row in importance.head(10).reset_index(drop=True).iterrows()))


1. Mean cell volume: 117 splits
2. Systolic blood pressure: 109 splits
3. HbA1c: 89 splits
4. Blood urea nitrogen: 77 splits
5. Glucose: 65 splits
6. Total cholesterol: 64 splits
7. Creatinine: 58 splits
8. White blood cell count: 51 splits
9. Lymphocyte %: 49 splits
10. C-reactive protein: 46 splits


![Feature importance](docs/images/project30_bioage/feature_importance.svg)

Split frequency is a model-specific predictive importance measure. It does **not** establish that changing a biomarker will change aging rate.


## 5. Biomarker-age gap diagnostics

For each person, we compute **predicted proxy age − chronological age**. Positive residuals mean the model sees a biomarker pattern that looks older than expected; negative residuals mean younger than expected. This is an exploratory residual, not a validated aging-acceleration endpoint.

![Age-gap distribution](docs/images/project30_bioage/age_gap_distribution.svg)


In [4]:
gap_df = test_df[["Age"] + FEATURES].copy()
gap_df["predicted_proxy_age"] = pt
gap_df["age_gap"] = pt - gap_df["Age"].to_numpy()
q20, q80 = gap_df["age_gap"].quantile([.2,.8])
low = gap_df[gap_df["age_gap"] <= q20]; high = gap_df[gap_df["age_gap"] >= q80]
rows=[]
for f in FEATURES:
    s=df[f].std(ddof=0)
    rows.append((f, low[f].mean(), high[f].mean(), (high[f].mean()-low[f].mean())/s))
contrast=pd.DataFrame(rows,columns=["feature","low_gap_mean","high_gap_mean","standardized_difference"])
contrast=contrast.reindex(contrast["standardized_difference"].abs().sort_values(ascending=False).index)
print(contrast.head(8).to_string(index=False))


1. Albumin: standardized high-vs-low gap difference 0.42
2. Blood urea nitrogen: standardized high-vs-low gap difference -0.30
3. Uric acid: standardized high-vs-low gap difference 0.22
4. Creatinine: standardized high-vs-low gap difference 0.17
5. HbA1c: standardized high-vs-low gap difference -0.12
6. White blood cell count: standardized high-vs-low gap difference 0.10
7. Lymphocyte %: standardized high-vs-low gap difference -0.08
8. C-reactive protein: standardized high-vs-low gap difference -0.07


## 6. Recreate the visual diagnostics in Python


In [ ]:
import matplotlib.pyplot as plt
pred = pt
gap = pred - yt
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(yt, pred, alpha=.35, s=18)
lo=min(yt.min(),pred.min()); hi=max(yt.max(),pred.max())
ax.plot([lo,hi],[lo,hi],"--")
ax.set(xlabel="Chronological age", ylabel="Biomarker-age proxy", title="Chronological vs biomarker-derived age proxy")
plt.show()

fig, ax = plt.subplots(figsize=(9,6))
top=importance.head(10).sort_values("split_count")
ax.barh(top["feature"], top["split_count"])
ax.set(title="Predictive split-frequency importance", xlabel="Tree split count")
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(gap, bins=20)
ax.axvline(0, linestyle="--")
ax.set(title="Age-gap distribution", xlabel="Predicted proxy age - chronological age", ylabel="People")
plt.show()


## 7. Longevity interpretation

### What this project can support
- population-level exploration of age-related biomarker patterns;
- ranking predictive signals worth deeper scientific investigation;
- benchmarking stronger biological-age models;
- designing future longitudinal or mortality-linked validation.

### What it cannot support
- diagnosing someone's “true biological age”;
- proving that a supplement, drug, diet or intervention slows aging;
- prescribing medical treatment from the model output;
- inferring causality from feature importance.


## 8. How a longevity company could extend this

A real longevity platform could replace chronological age as the only target with **longitudinal outcomes** such as mortality, frailty, disease-free survival, functional decline or repeated biomarker trajectories. The next engineering step would be calibration across cohorts, subgroup robustness, uncertainty estimates and prospective validation.


In [5]:
print("""PROJECT 30 — BIOAGE AI COMPLETE
============================================================
Dataset: NHANES-derived biomarker table
Adults with complete biomarker data: 4,552
Held-out test MAE: 9.74 years
Held-out test RMSE: 11.99 years
Held-out test R²: 0.50
Top model signals: Mean cell volume, Systolic blood pressure, HbA1c, Blood urea nitrogen, Glucose

Interpretation:
- This is a biomarker-derived AGE PROXY, not a clinical biological-age diagnosis.
- The age-gap output is exploratory and does not prove accelerated or slowed aging.
- Feature importance is predictive, not causal; it must not be used to prescribe anti-aging treatments.

PROJECT 30 COMPLETE ✅
""")

PROJECT 30 — BIOAGE AI COMPLETE
Dataset: NHANES-derived biomarker table
Adults with complete biomarker data: 4,552
Held-out test MAE: 9.74 years
Held-out test RMSE: 11.99 years
Held-out test R²: 0.50
Top model signals: Mean cell volume, Systolic blood pressure, HbA1c, Blood urea nitrogen, Glucose

Interpretation:
- This is a biomarker-derived AGE PROXY, not a clinical biological-age diagnosis.
- The age-gap output is exploratory and does not prove accelerated or slowed aging.
- Feature importance is predictive, not causal; it must not be used to prescribe anti-aging treatments.

PROJECT 30 COMPLETE ✅
